In [1]:
import json
from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
)


/home/aleksandr/Рабочий стол/ai-catalog-parser/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "numind/NuExtract3"

In [3]:
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(
    model_name
)

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:00<00:00, 880.68it/s] 


In [ ]:
image = Image.open("../data/pages/2026箭牌卫浴产品-5-9_page_2.png")

In [ ]:
schema = {
    "products": [
        {
            "name": "string",
            "description": ["string"],
            "product_bbox": {
                "x0": "number",
                "y0": "number",
                "x1": "number",
                "y1": "number",
            },
            "price": {
                "value": "number",
                "currency": "currency",
            },
            "specifications": {
                "dimensions": {
                    "length": "number",
                    "width": "number",
                    "height": "number",
                    "unit": "string",
                },
                "weight": {
                    "value": "number",
                    "unit": "string"
                },

                "other": ["string"],
            },
            "sku": "string",
        },
    ],
}

prompt = f"""
You are analyzing a catalog page.

Detect all product cards.

If there is no product card on the page, return ONLY an empty JSON.

Requirements:
- The “product_bbox” field is intended for the normalized coordinates of 
the product image's bounding box in the range [0, 1].

Otherwise, return ONLY valid JSON.

Schema:
{json.dumps(schema, indent=2)}
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image,
            },
            {
                "type": "text",
                "text": f"{prompt}",
            },
        ],
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)

generated_ids = model.generate(
    **inputs, 
    max_new_tokens=2048
)
generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(
        inputs.input_ids,
        generated_ids
    )
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


In [6]:
result = json.loads(output_text)
print(json.dumps(result, indent=4, ensure_ascii=False))

{
    "products": []
}
